# 🦎 Classifying threats with Laya

This page works through the three kinds of question that [Laya](https://huggingface.co/convaiinnovations/laya)
can answer, using one paragraph from a real listing rule. We then point the same questions at
every paragraph in a rule, and at a folder of rules.

Our example is the [final rule](https://www.federalregister.gov/d/2026-14636) that listed the
Florida Keys mole skink as endangered in July 2026. The skink lives in beach and coastal habitat
only a few feet above sea level, so it is a clear case of a species threatened by climate change.
It is a useful case for another reason too: as we will see, the rule rarely uses the words
"climate change".

## Setup

Laya runs as a web service. We send it a text and a set of questions, and it returns a probability
for each possible answer. The service address and your key are kept in a file called `.env` in your
project folder, which should never be committed to GitHub:

```
LAYA_URL=https://laya.carlboettiger.info
LAYA_API_KEY=your-key-here
```

`load_dotenv()` reads that file into environment variables. The `decide()` function sends one text
and a dictionary of questions, and returns Laya's answers.

In [1]:
import os
import re
import pathlib
import requests
import pandas as pd
from dotenv import load_dotenv

load_dotenv()
pd.set_option("display.max_colwidth", 150)
laya_url = os.environ["LAYA_URL"]
laya_auth = {"Authorization": f"Bearer {os.environ['LAYA_API_KEY']}"}

def decide(text, questions):
    response = requests.post(f"{laya_url}/v1/systemone", headers=laya_auth,
                             json={"state": text, "questions": questions}, timeout=120)
    response.raise_for_status()
    return response.json()["answers"]

## A text file

The Federal Register publishes the full text of every rule. We save the skink rule as a text file,
remove the HTML tags, and split it into paragraphs at blank lines. Any text file you have, such as a
report or a set of interview notes, can be read the same way.

In [2]:
rules = pathlib.Path("rules")
rules.mkdir(exist_ok=True)

def download(document_number):
    path = rules / f"{document_number}.txt"
    if not path.exists():
        meta = requests.get(f"https://www.federalregister.gov/api/v1/documents/{document_number}.json",
                            params={"fields[]": "raw_text_url"}, timeout=60).json()
        path.write_text(requests.get(meta["raw_text_url"], timeout=60).text)
    return path

def paragraphs(path):
    text = re.sub(r"<[^>]+>", "", path.read_text())
    paras = [re.sub(r"\s+", " ", p).strip() for p in re.split(r"\n\s*\n", text)]
    return [p for p in paras if len(p) > 200]

skink = paragraphs(download("2026-14636"))
len(skink)

45

Here is the paragraph that summarizes the threats to the skink:

In [3]:
example = next(p for p in skink if p.startswith("Sea level rise, more frequent tidal flooding"))
print(example)

Sea level rise, more frequent tidal flooding (increase of tides above the mean high tide), and increasing intensity of storm events (such as hurricanes) are the predominant threats to the Florida Keys mole skink and its habitat. Other threats to the Florida Keys mole skink include habitat loss and degradation that result from development and habitat disturbance. We also evaluated existing regulatory mechanisms and ongoing conservation measures. In the SSA report, we considered additional threats: overutilization due to recreational, educational, and scientific use; disease; oil spills; saltwater intrusion; and nonnative species. We concluded that, as indicated by the best available scientific and commercial information, these additional threats are currently having little to no impact on the Florida Keys mole skink, and thus their overall effect now and into the future is expected to be minimal. For full descriptions of all threats and how they impact the Florida Keys mole skink, pleas

## Noul questions

A `noul` question has a yes or no answer, and Laya returns the probability of yes. Notice that this
paragraph never uses the words "climate change", so we ask two versions of the question: one about
climate change and one about sea level rise.

In [4]:
decide(example, {
    "climate_threat": {"type": "noul",
                       "instructions": "Does this paragraph say that climate change threatens the species?"},
    "sea_level_threat": {"type": "noul",
                         "instructions": "Does this paragraph say that sea level rise threatens the species?"},
})

{'climate_threat': {'type': 'noul',
  'noul': 0.8568,
  'confidence': 0.8568,
  'answer_confidence': 0.8568,
  'action': {'act_probability': 1.0}},
 'sea_level_threat': {'type': 'noul',
  'noul': 0.8708,
  'confidence': 0.8708,
  'answer_confidence': 0.8708,
  'action': {'act_probability': 1.0}}}

Laya gives a high probability to both questions, so it connects sea level rise and storms to
climate change even though the paragraph does not name it. Delach et al. made the same kind of
judgment when they coded the rules by hand, and a regular expression for `climate change` would
miss this paragraph entirely.

## Choice questions

A `choice` question picks one option from a list that we define. Each option has a short
description in `criteria`, and Laya returns a probability for every option. The descriptions are
our coding scheme, written down so that it can be checked and changed.

Here we code two things. The first is what the paragraph is doing in the rule, which matters because
a rule also mentions climate change in background sections and in responses to public comments. The
second is which threat the paragraph names as the most important.

In [5]:
role_and_threat = {
    "role": {"type": "choice",
             "instructions": "What is this paragraph doing?",
             "criteria": {"threat finding": "concludes that a factor harms or will harm the species or its habitat",
                          "background": "general biological or legal background",
                          "dismissal": "considers a factor and concludes it is not a significant threat",
                          "comment response": "responds to a public or peer reviewer comment"}},
    "main_threat": {"type": "choice",
                    "instructions": "Which threat does this paragraph describe as the most important?",
                    "criteria": {"climate": "sea level rise, flooding, drought, heat or storms linked to a changing climate",
                                 "development": "habitat loss from building, roads or land conversion",
                                 "invasive species": "competition or predation by nonnative species",
                                 "harvest": "collection, hunting or other overuse",
                                 "disease": "pathogens or parasites",
                                 "none": "the paragraph does not describe a threat"}},
}
answers = decide(example, role_and_threat)
answers["role"]["choice"], answers["main_threat"]["choice"]

('threat finding', 'climate')

Each answer also holds the full set of probabilities, which shows how close the alternatives
came:

In [6]:
pd.DataFrame({name: answer["probabilities"] for name, answer in answers.items()})

,role,main_threat
threat finding,0.8506,NaN
background,0.0364,NaN
dismissal,0.0805,NaN
comment response,0.0326,NaN
climate,NaN,0.4859
development,NaN,0.2113
invasive species,NaN,0.0929
harvest,NaN,0.0528
disease,NaN,0.0834
none,NaN,0.0736


## Score questions

A `score` question places the text on an ordered scale. The `criteria` are a list of levels from
lowest to highest, and Laya returns a position along the scale that can fall between two levels.

Delach et al. coded two separate things for each species: whether climate change was named as a
threat, and whether any action to address it was planned. A score question lets us code each one by
degree instead of as yes or no.

In [7]:
scales = {
    "climate_weight": {"type": "score",
                       "instructions": "How much weight does this paragraph give to climate-related threats?",
                       "criteria": ["Climate-related threats are not mentioned.",
                                    "Climate-related threats are mentioned in passing.",
                                    "Climate-related threats are one of several important threats.",
                                    "Climate-related threats are the main threat."]},
    "mitigation": {"type": "score",
                   "instructions": "How specific are the actions this paragraph describes to reduce climate-related threats?",
                   "criteria": ["No actions are described.",
                                "Actions are mentioned in general terms.",
                                "Specific actions are described.",
                                "Specific actions are described with a schedule or a responsible agency."]},
}
answers = decide(example, scales)
{name: (round(answer["score"], 2), answer["legend"][str(round(answer["score"]))])
 for name, answer in answers.items()}

{'climate_weight': (2.23,
  'Climate-related threats are one of several important threats.'),
 'mitigation': (1.61, 'Specific actions are described.')}

The first score places climate among several important threats, which matches the paragraph.
The second score deserves a closer look. The paragraph mentions "ongoing conservation measures" but
describes no actions at all, so a mitigation score above 1 is probably wrong. A score is only as
good as our check of it, and checking means reading the text.

## A whole rule

The same questions can be asked of every paragraph. A single request sends one paragraph, and the
`/v1/systemone/batch` endpoint sends many paragraphs with the same questions in one request, which is
much faster. Here we keep only the paragraphs that mention a climate-related term, and code each of
them with all of the questions above.

In [8]:
def decide_many(texts, questions, size=32):
    answers = []
    for start in range(0, len(texts), size):
        response = requests.post(f"{laya_url}/v1/systemone/batch", headers=laya_auth,
                                 json={"states": texts[start:start + size], "questions": questions},
                                 timeout=300)
        response.raise_for_status()
        answers += [result["answers"] for result in response.json()["results"]]
    return answers

def code_paragraphs(texts):
    questions = role_and_threat | scales
    rows = []
    for text, answer in zip(texts, decide_many(texts, questions)):
        rows.append({"text": text[:150],
                     "role": answer["role"]["choice"],
                     "main_threat": answer["main_threat"]["choice"],
                     "climate_weight": round(answer["climate_weight"]["score"], 2),
                     "mitigation": round(answer["mitigation"]["score"], 2)})
    return pd.DataFrame(rows)

climate_terms = r"climat|sea level|flood|storm|drought"
coded = code_paragraphs([p for p in skink if re.search(climate_terms, p, re.I)])
coded

,text,role,main_threat,climate_weight,mitigation
0,"Why we need to publish a rule. Under the Act (16 U.S.C. 1531 et seq.), a species warrants listing if it meets the definition of an endangered spec...",threat finding,climate,2.15,1.74
1,"The final rule incorporates changes to our proposed listing rule and SSA report based on the comments we received, as discussed below in the Summa...",threat finding,none,1.84,1.56
2,(2) Comment: One commenter indicated that our current population resiliency estimates were too optimistic given the lack of available data regardi...,comment response,none,1.73,1.94
3,"and E). As outlined in our proposed rule (87 FR 58648, September 27, 2022), we determined that consideration of sea level rise impacts beyond 2060...",threat finding,climate,1.99,1.58
4,"coastal berms, rockland hammocks, and pine rocklands. However, individuals have been detected in developed areas such as cemeteries, vacant lots, ...",background,none,1.09,1.08
5,The SSA report documents the results of our comprehensive biological review of the best scientific and commercial data regarding the status of the...,background,none,1.27,1.84
6,"Sea level rise, more frequent tidal flooding (increase of tides above the mean high tide), and increasing intensity of storm events (such as hurri...",threat finding,climate,2.23,1.61
7,"The predominant threats currently affecting the Florida Keys mole skink and its habitat are sea level rise, more frequent high tide flooding, and ...",threat finding,climate,2.18,1.71
8,resiliency and overall species' redundancy. Long-term monitoring of the Florida Keys mole skink species and information on strong storm impacts to...,threat finding,climate,1.75,1.69
9,"and individual survival of Florida Keys mole skinks and, therefore, may result in a decline in population resiliency. Resiliency may be further re...",threat finding,development,1.97,1.97


## A folder of rules

To build a corpus, we download several rules into the same folder and code every file in it. The
second rule here lists the Peñasco least chipmunk, a mountain species in New Mexico.

In [9]:
download("2024-28338")

tables = []
for path in sorted(rules.glob("*.txt")):
    paras = [p for p in paragraphs(path) if re.search(climate_terms, p, re.I)]
    tables.append(code_paragraphs(paras).assign(rule=path.stem))
corpus = pd.concat(tables)
corpus.groupby(["rule", "role"]).size().unstack(fill_value=0)

role,background,comment response,dismissal,threat finding
rule,,,,
2024-28338,8,2,1,6
2026-14636,4,1,0,12


Before we count anything, we read the coded paragraphs from the chipmunk rule next to their
text:

In [10]:
corpus[corpus.rule == "2024-28338"].drop(columns="rule")

,text,role,main_threat,climate_weight,mitigation
0,"definition of critical habitat, as determined on the basis of the best scientific data available, are proposed for designation. However, section 4...",comment response,none,1.26,1.75
1,"significant risk factors for the subspecies and its habitat over the next 30-year period. In our proposed rule to list the species, we determined ...",threat finding,climate,1.86,1.85
2,The SSA report documents the results of our comprehensive biological review of the best scientific and commercial data regarding the status of the...,background,none,1.30,1.58
3,"of the species to adapt to both near-term and long-term changes in its physical and biological environment (for example, climate conditions, patho...",background,none,1.64,1.63
4,"To evaluate the current and future viability of the Pe[ntilde]asco least chipmunk, we assessed a range of conditions to allow us to consider the s...",comment response,invasive species,1.64,1.97
5,"We evaluated the past, current, and future stressors that affect the Pe[ntilde]asco least chipmunk's needs for long-term viability. Additionally, ...",background,development,1.82,2.03
6,result in changes to the plant community that do not adversely affect the prey base of the Mexican spotted owl but is detrimental to the specific ...,threat finding,invasive species,1.86,1.95
7,"for the White Mountains and the Sacramento Mountains (Service 2024, p. 52). It is suspected that the White Mountains and Sacramento Mountains popu...",threat finding,none,1.76,1.74
8,The range of the Pe[ntilde]asco least chipmunk once included the Sacramento and White Mountains in Lincoln and Otero Counties in New Mexico. The P...,dismissal,none,1.39,1.97
9,"In accordance with section 3(5)(A)(i) of the Act and regulations at 50 CFR 424.12(b), in determining which areas we will designate as critical hab...",background,none,1.79,2.00


## Where Laya goes wrong

Many of these answers are poor, and finding them is the most useful thing this page can teach. We
list some of the problems below. We do not give the fixes, because working them out is part of the
module.

- **Procedure coded as a threat.** In the skink rule, the paragraph that explains why the agency must
  publish a rule, and the paragraph that says the final rule incorporates public comments, are both
  coded as threat findings. So is a paragraph that says which future scenarios the agency chose not
  to present.
- **Paragraphs that should never have been coded.** Several descriptions of critical habitat units in
  the chipmunk rule are coded as threat findings. Check which climate term each of them matched.
- **A dismissal that is missed.** One chipmunk paragraph says that climate change is expected to have
  a low impact on the species. Find it, and compare its codes with what the paragraph says.
- **Scores that barely move.** Paragraphs that never mention climate still receive a climate weight
  near the middle of the scale, and almost every paragraph receives a similar mitigation score,
  including legal descriptions of map units.
- **Choices forced onto the wrong options.** Some methods paragraphs are coded as comment responses,
  and some are assigned a main threat that they never discuss.
- **Paragraphs that are not paragraphs.** Some rows begin mid-sentence, and others carry a page
  header or a section heading glued to their first line.

## Directions

A stronger model such as Jev may make fewer of these mistakes, but it would not tell us which ones it
made, and it would leave our coding scheme as vague as it is now. These questions point toward
improvements we can measure.

- **What does Laya see?** Each request contains one block of text and nothing else. Laya does not
  know the species, the section of the rule, or what came before. The rules have section headings,
  and the Federal Register API offers more than one format of each document.
- **What passes the filter?** Read a sample of what a regular expression returns before you
  classify any of it.
- **Are the options complete?** A `choice` question must pick one of the options it is given. Ask
  what Laya should do with a paragraph that fits none of them.
- **Are the descriptions clear?** The `criteria` are the only definition Laya has of each option.
  Change one description at a time and measure which answers move.
- **Is the question too broad?** One question can sometimes be split into several narrow `noul`
  questions whose answers we combine in code.
- **How should a score be read?** Look at the distribution of a score across many paragraphs
  before you interpret any single value. The `confidence` field can help decide which paragraphs
  need a person to read them.
- **How often is Laya right?** Label a sample by hand and measure agreement. With enough labels, Laya
  can also be [fine-tuned](https://github.com/NandhaKishorM/laya/blob/main/notebooks/laya_finetune_typed_decisions_2xT4_kaggle.ipynb)
  on your own coding scheme.

See the [Laya documentation](https://nandhakishorm.github.io/laya/) and the
[Jev documentation](https://docs.typesafe.ai/primitives) for the full question format.